## 1. Train GIST-LIKE + XGBOOST

In [1]:
import os
import cv2
import joblib
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

# CONFIG
gist_csv = r"D:\alzheimer detection.v1i.folder\Fitur_Ekstraksi_Klasik\features_extracted\features_gist_like_multiblock.csv"
model_dir = r"D:\alzheimer detection.v1i.folder\Dashboard\src\classical\model"
os.makedirs(model_dir, exist_ok=True)
save_path = os.path.join(model_dir, "gist_xgb_model.pkl")

IMG_SIZE = 224
GRID = 4  # 4x4 blocks

# GIST-LIKE FEATURE EXTRACTOR
def extract_gist_like(image):
    """
    image: np.ndarray RGB
    return: 1D feature vector
    """
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    gray = cv2.resize(gray, (IMG_SIZE, IMG_SIZE))

    h, w = gray.shape
    bh, bw = h // GRID, w // GRID

    features = []
    for i in range(GRID):
        for j in range(GRID):
            block = gray[i*bh:(i+1)*bh, j*bw:(j+1)*bw]
            features.append(block.mean())

    return np.array(features, dtype=np.float32)

# LOAD DATASET
if not os.path.exists(gist_csv):
    raise FileNotFoundError("CSV GIST tidak ditemukan")

df = pd.read_csv(gist_csv)

numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
X = df[numeric_cols].fillna(0).values
y = df["label"]

# ENCODE & SCALE
le = LabelEncoder()
y_enc = le.fit_transform(y)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# SPLIT
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X_scaled, y_enc, test_size=0.2, stratify=y_enc, random_state=42
)

X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.25, stratify=y_trainval, random_state=42
)

# TRAIN XGBOOST
model = XGBClassifier(
    n_estimators=500,
    max_depth=7,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="mlogloss",
    random_state=42
)

model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_val, y_val)],
    verbose=False
)

# SAVE FULL PIPELINE
joblib.dump({
    "model": model,
    "scaler": scaler,
    "label_encoder": le,
    "feature_extractor": extract_gist_like,
    "img_size": IMG_SIZE,
    "grid": GRID
}, save_path)

print("\n[SUCCESS] GIST XGBoost with image extractor saved")
print("Feature dimension:", X.shape[1])


[SUCCESS] GIST XGBoost with image extractor saved
Feature dimension: 768


## 2. Train HOG + XGBOOST

In [ ]:
import os
import cv2
import joblib
import numpy as np
import pandas as pd

from skimage.feature import hog
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

# CONFIG
hog_csv = r"D:\alzheimer detection.v1i.folder\Fitur_Ekstraksi_Klasik\features_extracted\features_hog.csv"
model_dir = r"D:\alzheimer detection.v1i.folder\Dashboard\src\classical\model"
os.makedirs(model_dir, exist_ok=True)
save_path = os.path.join(model_dir, "hog_xgb_model.pkl")

IMG_SIZE = 128
HOG_PARAMS = dict(
    orientations=9,
    pixels_per_cell=(8, 8),
    cells_per_block=(2, 2),
    block_norm="L2-Hys"
)

# HOG FEATURE EXTRACTOR
def extract_hog(image):
    """
    image: np.ndarray RGB
    return: 1D HOG feature vector
    """
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    gray = cv2.resize(gray, (IMG_SIZE, IMG_SIZE))

    feat = hog(
        gray,
        orientations=HOG_PARAMS["orientations"],
        pixels_per_cell=HOG_PARAMS["pixels_per_cell"],
        cells_per_block=HOG_PARAMS["cells_per_block"],
        block_norm=HOG_PARAMS["block_norm"],
        feature_vector=True
    )

    return feat.astype(np.float32)

# LOAD DATASET
if not os.path.exists(hog_csv):
    raise FileNotFoundError("CSV HOG tidak ditemukan")

df = pd.read_csv(hog_csv)

numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
X = df[numeric_cols].fillna(0).values
y = df["label"]

# ENCODE & SCALE
le = LabelEncoder()
y_enc = le.fit_transform(y)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# SPLIT
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X_scaled, y_enc, test_size=0.2, stratify=y_enc, random_state=42
)

X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.25, stratify=y_trainval, random_state=42
)

# TRAIN XGBOOST
model = XGBClassifier(
    n_estimators=500,
    max_depth=7,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="mlogloss",
    random_state=42
)

model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_val, y_val)],
    verbose=False
)

# VALIDATION DIM CHECK (PENTING!)
dummy = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
hog_dim = extract_hog(dummy).shape[0]

if hog_dim != X.shape[1]:
    raise RuntimeError(
        f"[DIM ERROR] HOG dim mismatch: extractor={hog_dim}, CSV={X.shape[1]}"
    )

# SAVE FULL PIPELINE
joblib.dump({
    "model": model,
    "scaler": scaler,
    "label_encoder": le,
    "feature_extractor": extract_hog,
    "hog_params": HOG_PARAMS,
    "img_size": IMG_SIZE
}, save_path)

print("\n[SUCCESS] HOG XGBoost with image extractor saved")
print("Feature dimension:", hog_dim)

c:\Python312\Lib\site-packages\xgboost\training.py:199: UserWarning: [23:41:02] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



[SUCCESS] Model HOG XGBoost berhasil disimpan
Path model : D:\alzheimer detection.v1i.folder\Dashboard\src\classical\model\hog_xgb_model.pkl
Total data : 9766
Dimensi fitur HOG : 8100


## 3. Train Hu moment + XGBOOST

In [ ]:
import os
import cv2
import joblib
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

# CONFIG
hu_csv = r"D:\alzheimer detection.v1i.folder\Fitur_Ekstraksi_Klasik\features_extracted\features_hu_moments_multi_patch.csv"
model_dir = r"D:\alzheimer detection.v1i.folder\Dashboard\src\classical\model"
os.makedirs(model_dir, exist_ok=True)
save_path = os.path.join(model_dir, "hu_xgb_model.pkl")

IMG_SIZE = 256
GRID = 4  # 4x4 patches → 16 patches
HU_DIM = 7

# HU MOMENTS MULTI-PATCH EXTRACTOR
def extract_hu_multipatch(image):
    """
    image: np.ndarray RGB
    return: 1D Hu Moments feature vector
    """
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    gray = cv2.resize(gray, (IMG_SIZE, IMG_SIZE))

    h, w = gray.shape
    ph, pw = h // GRID, w // GRID

    features = []

    for i in range(GRID):
        for j in range(GRID):
            patch = gray[i*ph:(i+1)*ph, j*pw:(j+1)*pw]
            moments = cv2.moments(patch)
            hu = cv2.HuMoments(moments).flatten()

            # log transform (standar HU Moments)
            hu = -np.sign(hu) * np.log10(np.abs(hu) + 1e-10)
            features.extend(hu)

    return np.array(features, dtype=np.float32)

# LOAD DATASET
if not os.path.exists(hu_csv):
    raise FileNotFoundError("CSV HU Moments tidak ditemukan")

df = pd.read_csv(hu_csv)

numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
X = df[numeric_cols].fillna(0).values
y = df["label"]

# ENCODE & SCALE
le = LabelEncoder()
y_enc = le.fit_transform(y)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# SPLIT
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X_scaled, y_enc, test_size=0.2, stratify=y_enc, random_state=42
)

X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.25, stratify=y_trainval, random_state=42
)

# TRAIN XGBOOST
model = XGBClassifier(
    n_estimators=500,
    max_depth=7,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="mlogloss",
    random_state=42
)

model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_val, y_val)],
    verbose=False
)

# VALIDATION DIM CHECK (WAJIB!)
dummy = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
hu_dim = extract_hu_multipatch(dummy).shape[0]

if hu_dim != X.shape[1]:
    raise RuntimeError(
        f"[DIM ERROR] HU dim mismatch: extractor={hu_dim}, CSV={X.shape[1]}"
    )

# SAVE FULL PIPELINE
joblib.dump({
    "model": model,
    "scaler": scaler,
    "label_encoder": le,
    "feature_extractor": extract_hu_multipatch,
    "img_size": IMG_SIZE,
    "grid": GRID
}, save_path)

print("\n[SUCCESS] HU Moments XGBoost with image extractor saved")
print("Feature dimension:", hu_dim)

c:\Python312\Lib\site-packages\xgboost\training.py:199: UserWarning: [00:28:07] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



[SUCCESS] Model HU Moments XGBoost berhasil disimpan
Path model : D:\alzheimer detection.v1i.folder\Dashboard\src\classical\model\hu_xgb_model.pkl
Total data : 9766
Dimensi fitur HU Moments : 112
